In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pygris import tracts 
from pygris.data import get_lodes
import matplotlib.pyplot as plt



In [ ]:
NY_2009_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2009.csv")
NY_2010_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2010.csv")
NY_2011_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2011.csv")
NY_2012_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2012.csv")
NY_2013_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2013.csv")
NY_2014_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2014.csv")
NY_2015_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2015.csv")
NY_2016_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2016.csv")
NY_2017_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2017.csv")
NY_2018_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2018.csv")
NY_2019_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2019.csv")
NY_2020_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2020.csv")
#NY_2022_commute_df=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_od_main_JT01_2022.csv")
crosswalk=pd.read_csv("D:/LMU_Life/Housing/project/LODES Data/ny_xwalk.csv")

In [ ]:
years = [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

tract_list=[]

# === Loop through each year ===
for year in years:
    # Load data
    df = globals()[f"NY_{year}_commute_df"]

    tracts_in_a_year=pd.DataFrame({'tract': np.unique(df[['w_geocode', "h_geocode"]].values)})
    tract_list.append(tracts_in_a_year)
    
all_tracts=pd.concat(tract_list).drop_duplicates().reset_index(drop=True)

all_tracts

years_df = pd.DataFrame({"year": years})

# Cross join: every tract × every year
tract_year_grid = all_tracts.merge(years_df, how="cross")

In [ ]:
aggregated_by_year_work = {}

# === Loop through each year ===
for year in years:
    # Load data
    df = globals()[f"NY_{year}_commute_df"]

    # Apply the same aggregation logic
    aggregated_data = (
        df.groupby("w_geocode")
          .agg(
              Flow_to_work=("S000", "sum"),
              SA01_to_work=("SA01", "sum"),
              SA02_to_work=("SA02", "sum"),
              SA03_to_work=("SA03", "sum"),
              SE01_to_work=("SE01", "sum"),
              SE02_to_work=("SE02", "sum"),
              SE03_to_work=("SE03", "sum"),
              SI01_to_work=("SI01", "sum"),
              SI02_to_work=("SI02", "sum"),
              SI03_to_work=("SI03", "sum"),
          )
    )
    # Add the year column
    #aggregated_data["year"] = year
    
    # Store in dictionary
    aggregated_by_year_work[year] = aggregated_data


aggregated_by_year_home = {}

# === Loop through each year ===
for year in years:
    # Load data
    df = globals()[f"NY_{year}_commute_df"]

    # Apply the same aggregation logic
    aggregated_data = (
        df.groupby("h_geocode")
          .agg(
                     Flow_to_home=("S000", "sum"),
                     SA01_to_home=("SA01", "sum"),
                     SA02_to_home=("SA02", "sum"),
                     SA03_to_home=("SA03", "sum"),
                     SE01_to_home=("SE01", "sum"),
                     SE02_to_home=("SE02", "sum"),
                     SE03_to_home=("SE03", "sum"),
                     SI01_to_home=("SI01", "sum"),
                     SI02_to_home=("SI02", "sum"),
                     SI03_to_home=("SI03", "sum"),
                 ))
    
    # Add the year column
    #aggregated_data["year"] = year
    
    # Store in dictionary
    aggregated_by_year_home[year] = aggregated_data

   

In [ ]:
panel_df_work = pd.concat(aggregated_by_year_work, names=["year"])
panel_df_work = panel_df_work.reset_index()

panel_df_home = pd.concat(aggregated_by_year_home, names=["year"])
panel_df_home = panel_df_home.reset_index()

In [ ]:
tract_year_grid_work=pd.merge(tract_year_grid, panel_df_work, how="left", left_on=["tract", "year"], right_on=["w_geocode", "year"])
tract_year_grid_work_home=pd.merge(tract_year_grid_work, panel_df_home, how="left", left_on=["tract", "year"], right_on=["h_geocode", "year"])
tract_year_grid_work_home

In [ ]:
tract_year_grid_work_home=(
    tract_year_grid_work_home
    .assign(trct = tract_year_grid_work_home["tract"]
            .astype(str)
            .str.slice(stop=11))
)

tract_year_grid_work_home